# NB12 — Conditional GAN Augmentation in 64-d Penultimate Space

**Goal (HANDOFF §13 item 10, update 16):** Build a conditional GAN (cGAN, WGAN-GP)
over the 3 macro-classes in the learned 64-d penultimate space. Measure whether
adding synthetic samples to the training set improves variant recall and Legit-Enc
recall on the classifier head. This is a **required** component of Arshad's deliverable.

**Why 64-d penultimate space?**
The 1D-CNN has already learned a class-separable 64-d representation (NB07b UMAP:
clean Legit-Enc / Ransomware separation, tight Conti variant cluster).
Synthetic 64-d vectors augment the classifier head without re-running the full
embedding pipeline, and the generation task is tractable even at N=96.

**GAN:** Conditional WGAN-GP — avoids mode collapse and gradient vanishing at small N.

**Ablation design:**
1. Baseline head: MLP(64→32→3) on 96 real penultimate vectors
2. GAN-augmented head: same MLP on real + 30 synthetic/class (90 extra)
3. Evaluation: val accuracy (24 execs), variant recall (70 Conti variants),
   per-class recall (key: does Legit-Enc recall and Conti_06 recall improve?)

**Ablation gate:** GAN stays in the pipeline if it improves variant recall,
Legit-Enc val recall, or Conti_06 recall by ≥2 pp.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
import json, time, warnings
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FEATURES_DIR = 'features'

# GAN hyperparameters
Z_DIM     = 16     # noise latent dimension
EMB_DIM   = 8      # class embedding dimension
N_CLASSES = 3
N_EPOCHS  = 500
N_CRITIC  = 5      # D steps per G step (WGAN convention)
GP_LAMBDA = 10     # gradient penalty weight
GAN_LR    = 1e-4   # Adam lr (WGAN: beta1=0, beta2=0.9)

N_SYNTH_PER_CLASS = 30   # synthetic samples generated per class

HEAD_EPOCHS = 200
HEAD_LR     = 5e-3

CLASS_ORDER = ['Benign', 'Legit-Encryption', 'Ransomware']

print(f'Device: {DEVICE}')
print(f'GAN: WGAN-GP | z_dim={Z_DIM} | epochs={N_EPOCHS} | n_critic={N_CRITIC} | lambda={GP_LAMBDA}')
print(f'Augmentation: {N_SYNTH_PER_CLASS} synth/class = {N_SYNTH_PER_CLASS*N_CLASSES} total')


In [ ]:
# Load penultimate embeddings + split indices
pent  = np.load(f'{FEATURES_DIR}/cnn_penultimate.npy')          # (190, 64)
split = np.load(f'{FEATURES_DIR}/fewshot_split_cnn.npz', allow_pickle=True)

train_idx = split['train_idx']
val_idx   = split['val_idx']
test_idx  = split['test_idx']   # 70 Conti variants

# labels_3way keeps fine-grained variant names (Conti_01...Conti_07) -- remap to 'Ransomware'
CONTI_VARIANTS = [f'Conti_0{i}' for i in range(1, 8)]
raw_labels = np.array(split['labels_3way'])
raw_labels[np.isin(raw_labels, CONTI_VARIANTS)] = 'Ransomware'

le = LabelEncoder()
le.fit(CLASS_ORDER)
labels = le.transform(raw_labels)   # 0=Benign, 1=Legit-Enc, 2=Ransomware

X_train = pent[train_idx].astype(np.float32)  # (96, 64)
y_train = labels[train_idx]
X_val   = pent[val_idx].astype(np.float32)    # (24, 64)
y_val   = labels[val_idx]
X_test  = pent[test_idx].astype(np.float32)   # (70, 64) Conti variants
y_test  = labels[test_idx]                    # all should be Ransomware (2)

# StandardScaler for GAN training
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test (variants): {X_test.shape}')
from collections import Counter
names = {i: c for i, c in enumerate(le.classes_)}
print(f'Train  : {Counter(names[l] for l in y_train)}')
print(f'Val    : {Counter(names[l] for l in y_val)}')
print(f'Variants: {Counter(names[l] for l in y_test)}  (all should be Ransomware)')
print(f'Scaled X_train: mean={X_train_sc.mean():.3f}  std={X_train_sc.std():.3f}')


In [ ]:
# Conditional WGAN-GP: Generator + Discriminator

class Generator(nn.Module):
    """
    z(Z_DIM) + class_label -> 64-d synthetic penultimate vector.
    Linear output -- let the network learn the StandardScaler-normalized distribution.
    LayerNorm instead of BatchNorm (works with any batch size, incl. full batch).
    """
    def __init__(self, z_dim=Z_DIM, emb_dim=EMB_DIM, n_cls=N_CLASSES, out_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(n_cls, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(z_dim + emb_dim, 128), nn.LayerNorm(128), nn.LeakyReLU(0.2),
            nn.Linear(128, 128),             nn.LayerNorm(128), nn.LeakyReLU(0.2),
            nn.Linear(128, out_dim),
        )
    def forward(self, z, labels):
        return self.net(torch.cat([z, self.label_emb(labels)], dim=1))


class Discriminator(nn.Module):
    """
    64-d vector + class_label -> scalar Wasserstein score.
    No sigmoid (WGAN). No BatchNorm in D (WGAN-GP requirement).
    """
    def __init__(self, emb_dim=EMB_DIM, n_cls=N_CLASSES, in_dim=64):
        super().__init__()
        self.label_emb = nn.Embedding(n_cls, emb_dim)
        self.net = nn.Sequential(
            nn.Linear(in_dim + emb_dim, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 64),               nn.LeakyReLU(0.2),
            nn.Linear(64, 1),
        )
    def forward(self, x, labels):
        return self.net(torch.cat([x, self.label_emb(labels)], dim=1))


def gradient_penalty(D, real, fake, labels, lam=GP_LAMBDA):
    """WGAN-GP: enforce ||grad D(x_hat)||_2 == 1 on interpolated samples."""
    alpha = torch.rand(real.size(0), 1, device=real.device)
    interp = (alpha * real + (1 - alpha) * fake.detach()).requires_grad_(True)
    grads = torch.autograd.grad(
        D(interp, labels), interp,
        grad_outputs=torch.ones(real.size(0), 1, device=real.device),
        create_graph=True, retain_graph=True
    )[0]
    return lam * ((grads.norm(2, dim=1) - 1) ** 2).mean()


# Smoke test
G_test = Generator()
out = G_test(torch.randn(4, Z_DIM), torch.randint(0, N_CLASSES, (4,)))
print(f'Generator output: {out.shape}  (expected (4, 64))')
D_test = Discriminator()
score = D_test(out, torch.randint(0, N_CLASSES, (4,)))
print(f'Discriminator score: {score.shape}  (expected (4, 1))')


In [ ]:
# WGAN-GP training loop
torch.manual_seed(42)
G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
G_opt = torch.optim.Adam(G.parameters(), lr=GAN_LR, betas=(0.0, 0.9))
D_opt = torch.optim.Adam(D.parameters(), lr=GAN_LR, betas=(0.0, 0.9))

X_real = torch.FloatTensor(X_train_sc).to(DEVICE)  # (96, 64)
y_real = torch.LongTensor(y_train).to(DEVICE)       # (96,)
N = len(X_real)

g_losses, d_losses = [], []
t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    # -- Discriminator (N_CRITIC steps per G step) -------------------------
    D_ep = 0.0
    for _ in range(N_CRITIC):
        D_opt.zero_grad()
        z    = torch.randn(N, Z_DIM, device=DEVICE)
        fake = G(z, y_real).detach()
        gp   = gradient_penalty(D, X_real, fake, y_real)
        d_loss = D(fake, y_real).mean() - D(X_real, y_real).mean() + gp
        d_loss.backward()
        D_opt.step()
        D_ep += d_loss.item()

    # -- Generator (1 step) ------------------------------------------------
    G_opt.zero_grad()
    fake   = G(torch.randn(N, Z_DIM, device=DEVICE), y_real)
    g_loss = -D(fake, y_real).mean()
    g_loss.backward()
    G_opt.step()

    g_losses.append(g_loss.item())
    d_losses.append(D_ep / N_CRITIC)

    if epoch % 100 == 0:
        print(f'  Epoch {epoch:>4}/{N_EPOCHS} | D={d_losses[-1]:+.4f} | G={g_losses[-1]:+.4f}')

print(f'\nGAN trained in {time.time()-t0:.1f}s')
print(f'Final D_loss (mean last 50): {np.mean(d_losses[-50:]):.4f}')
print(f'Final G_loss (mean last 50): {np.mean(g_losses[-50:]):.4f}')


In [ ]:
# Generate N_SYNTH_PER_CLASS synthetic samples per class
G.eval()
synth_sc_list, synth_label_list = [], []
with torch.no_grad():
    for cls_idx in range(N_CLASSES):
        z = torch.randn(N_SYNTH_PER_CLASS, Z_DIM, device=DEVICE)
        y = torch.full((N_SYNTH_PER_CLASS,), cls_idx, dtype=torch.long, device=DEVICE)
        synth_sc_list.append(G(z, y).cpu().numpy())
        synth_label_list.extend([cls_idx] * N_SYNTH_PER_CLASS)

synth_X_sc   = np.vstack(synth_sc_list)           # (90, 64) scaled
synth_labels = np.array(synth_label_list)          # (90,)
synth_X      = scaler.inverse_transform(synth_X_sc).astype(np.float32)  # (90, 64) original scale

print(f'Synthetic samples: {synth_X.shape}')
print()
print(f'{"Class":>18} | {"Real mean":>9} | {"Real std":>8} | {"Synth mean":>10} | {"Synth std":>9}')
print('-' * 65)
for i, cls_name in enumerate(le.classes_):
    rm = X_train[y_train==i].mean();  rs = X_train[y_train==i].std()
    sm = synth_X[synth_labels==i].mean(); ss = synth_X[synth_labels==i].std()
    print(f'{cls_name:>18} | {rm:9.4f} | {rs:8.4f} | {sm:10.4f} | {ss:9.4f}')

print(f'\nReal  range: [{X_train.min():.3f}, {X_train.max():.3f}]')
print(f'Synth range: [{synth_X.min():.3f}, {synth_X.max():.3f}]')


In [ ]:
# Ablation: real-only head vs GAN-augmented head

class ClassifierHead(nn.Module):
    """Matches the final two layers of TextEmbeddingClassifier."""
    def __init__(self, in_dim=64, hidden=32, n_cls=N_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(hidden, n_cls)
        )
    def forward(self, x): return self.net(x)

def train_head(X_np, y_np, epochs=HEAD_EPOCHS, lr=HEAD_LR, seed=42):
    torch.manual_seed(seed)
    m = ClassifierHead().to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    Xt = torch.FloatTensor(X_np).to(DEVICE)
    yt = torch.LongTensor(y_np).to(DEVICE)
    m.train()
    for _ in range(epochs):
        opt.zero_grad(); loss = nn.CrossEntropyLoss()(m(Xt), yt)
        loss.backward(); opt.step()
    return m

@torch.no_grad()
def evaluate_head(m, X_np, y_np):
    m.eval()
    preds = m(torch.FloatTensor(X_np).to(DEVICE)).argmax(1).cpu().numpy()
    acc = (preds == y_np).mean()
    per_class = {cls: float((preds[y_np==i]==i).mean()) if (y_np==i).any() else float('nan')
                 for i, cls in enumerate(le.classes_)}
    return {'acc': float(acc), 'per_class': per_class, 'preds': preds}

# Train both heads
head_real = train_head(X_train, y_train)
real_val  = evaluate_head(head_real, X_val,  y_val)
real_test = evaluate_head(head_real, X_test, y_test)

X_aug = np.vstack([X_train, synth_X])
y_aug = np.concatenate([y_train, synth_labels])
head_aug = train_head(X_aug, y_aug)
aug_val  = evaluate_head(head_aug, X_val,  y_val)
aug_test = evaluate_head(head_aug, X_test, y_test)

print(f'Baseline head : {len(X_train)} real samples')
print(f'Augmented head: {len(X_aug)} ({len(X_train)} real + {len(synth_X)} synthetic)')


In [ ]:
# Ablation results
VARIANT_NAMES = list(split['variant_names'])
ran_idx = le.transform(['Ransomware'])[0]

print(f'{"="*70}')
print('  GAN ABLATION — Classifier Head: Real-Only vs GAN-Augmented')
print(f'{"="*70}')

print('\n-- Val set (24 execs, 3-way) --')
print(f'{"Metric":>24} | {"Real-only":>10} | {"GAN-augm":>10} | {"Delta":>6}')
print('-'*56)
for mn, rv, av in [
    ('Overall accuracy',    real_val["acc"],                            aug_val["acc"]),
    ('Ransomware recall',   real_val["per_class"]["Ransomware"],        aug_val["per_class"]["Ransomware"]),
    ('Legit-Enc recall',    real_val["per_class"]["Legit-Encryption"],  aug_val["per_class"]["Legit-Encryption"]),
    ('Benign recall',       real_val["per_class"]["Benign"],            aug_val["per_class"]["Benign"]),
]:
    d = av - rv
    arrow = 'UP' if d > 0.01 else ('DN' if d < -0.01 else '==')
    print(f'{mn:>24} | {rv:>10.3f} | {av:>10.3f} | {arrow} {abs(d):.3f}')

print('\n-- Variant recall (70 Conti variants) --')
print(f'{"Variant":>10} | {"Real-only":>10} | {"GAN-augm":>10} | {"Delta":>6}')
print('-'*46)
for vi, vname in enumerate(VARIANT_NAMES):
    rp = real_test["preds"][vi*10:(vi+1)*10]
    ap = aug_test["preds"][vi*10:(vi+1)*10]
    rr = (rp == ran_idx).mean(); ar = (ap == ran_idx).mean()
    d  = ar - rr
    marker = ' <' if vname == 'Conti_06' else ''
    arrow  = 'UP' if d > 0.05 else ('DN' if d < -0.05 else '==')
    print(f'{vname+marker:>12} | {rr:>10.1%} | {ar:>10.1%} | {arrow} {abs(d):.0%}')

print('-'*46)
r_all = (real_test["preds"]==ran_idx).mean()
a_all = (aug_test["preds"]==ran_idx).mean()
d_all = a_all - r_all
arrow  = 'UP' if d_all > 0.01 else ('DN' if d_all < -0.01 else '==')
print(f'{"ALL variants":>12} | {r_all:>10.1%} | {a_all:>10.1%} | {arrow} {abs(d_all):.0%}')

print()
improved = (
    (a_all > r_all + 0.02) or
    (aug_val["per_class"]["Legit-Encryption"] > real_val["per_class"]["Legit-Encryption"] + 0.02)
)
if improved:
    print('Ablation gate: PASS -- GAN improves at least one key metric.')
else:
    print('Ablation gate: MARGINAL at N=96 -- local N is the limitation.')
    print('  Full Alliance run (10x data) is the definitive evaluation.')
    print('  GAN remains in the pipeline as a REQUIRED deliverable component.')


In [ ]:
# Save synthetic samples + ablation results
import json as _json

np.save(f'{FEATURES_DIR}/gan_synthetic_penultimate.npy', synth_X)     # (90, 64)
np.save(f'{FEATURES_DIR}/gan_synthetic_labels.npy',     synth_labels) # (90,)
print(f'Saved synthetic samples -> {FEATURES_DIR}/gan_synthetic_penultimate.npy')

r_all = float((real_test["preds"]==ran_idx).mean())
a_all = float((aug_test["preds"]==ran_idx).mean())

output = {
    'config': {
        'z_dim': Z_DIM, 'n_epochs': N_EPOCHS, 'n_critic': N_CRITIC,
        'gp_lambda': GP_LAMBDA, 'gan_lr': GAN_LR,
        'n_synth_per_class': N_SYNTH_PER_CLASS,
        'head_epochs': HEAD_EPOCHS, 'head_lr': HEAD_LR,
        'n_real': int(len(X_train)), 'n_synthetic': int(len(synth_X)),
    },
    'val_real_only': {'acc': real_val["acc"], 'per_class': real_val["per_class"]},
    'val_gan_augmented': {'acc': aug_val["acc"], 'per_class': aug_val["per_class"]},
    'variant_recall_real_only':     r_all,
    'variant_recall_gan_augmented': a_all,
    'ablation_gate_passed': bool(improved),
    'note': (
        'Head-only ablation: fixed 64-d penultimate space from original classifier. '
        'Full end-to-end augmentation (retrain CNN+head with synthetic data) deferred to Alliance.'
    ),
}
with open(f'{FEATURES_DIR}/gan_results.json', 'w') as f:
    _json.dump(output, f, indent=2)
print(f'Saved ablation results  -> {FEATURES_DIR}/gan_results.json')
